# 3D.01 — Strategy Rules & Position Construction

## 1. Research Objective

This notebook translates the predefined 12-month time-series momentum signal into systematic Long/Short/Flat positions across 16 ETFs.

The objective is to establish explicit trading rules, temporal alignment conventions, and position validation procedures before economic backtesting.

Only Train (2010–2017) and Validation (2018–2021) data are used. The Test period (2022–2025) remains untouched until Phase 3F.

## 2. Data & Research Environment

This section initializes the research environment and reuses the validated data preparation and momentum signal functions developed in Phase 3C.

In [54]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np 

project_root = Path.cwd()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.signals

from src.research import prepare_momentum_samples
from src.data import download_prices, to_monthly_prices, validate_monthly_prices
from src.signals import calculate_momentum, generate_signals, clean_signals_and_targets, generate_targets

In [55]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

start_date = "2008-01-01"
end_date = "2022-01-01"
lookback = 12

prices = download_prices(tickers, start_date, end_date, auto_adjust=True, progress=False)

monthly_prices = to_monthly_prices(prices)

print(monthly_prices.tail())
print(monthly_prices.shape)
print(monthly_prices.isna().sum().sum())

validate_monthly_prices(monthly_prices, tickers, start_date, end_date)

print("Monthly prices validation passed.")



Ticker            DBA        DBB        DBC        EEM        EWJ        FXA  \
Date                                                                           
2021-08-31  16.881031  17.838554  16.689245  46.598080  60.735088  69.112045   
2021-09-30  16.872208  17.727222  17.559383  44.793201  62.377773  68.303352   
2021-10-31  17.022221  18.343822  18.577448  45.273315  60.735088  71.043419   
2021-11-30  17.083994  17.898500  16.950287  43.423973  58.897053  67.285271   
2021-12-31  17.428143  19.063189  18.081467  44.078400  60.238152  68.588768   

Ticker             FXB         FXE        FXY        HYG         IEF  \
Date                                                                   
2021-08-31  119.405846  104.293381  85.500000  65.650490  100.016312   
2021-09-30  116.999146  102.201439  84.459999  65.407524   98.418938   
2021-10-31  118.774872  101.889084  82.459999  65.204239   97.983032   
2021-11-30  115.367638   99.863411  83.099998  64.442001   99.053108   
2021-12

## 3. Momentum Signal Construction

The strategy uses a predefined 12-month time-series momentum signal.

- Positive momentum: +1 (Long signal).
- Negative momentum: -1 (Short signal).
- Zero momentum: 0 (Flat signal).
- Insufficient historical data: NaN.

The first 12 monthly observations are undefined because the momentum calculation requires 12 months of historical returns.

In [56]:
momentum_12m = calculate_momentum(monthly_prices, lookback=12)

signals_12m = generate_signals(momentum_12m)

print(signals_12m.head(15))

Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2008-01-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-02-29  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-03-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-04-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-05-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-06-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-07-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-08-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-09-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-10-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-11-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

## 4. Position Construction & Validation

Trading positions are obtained by lagging momentum signals by one month using `shift(1)`.

This ensures that a signal computed at the end of month t is applied to the return of month t+1.

The first 13 monthly positions are undefined. Missing positions are preserved as NaN rather than automatically interpreted as Flat positions.

Valid position values are -1, 0, +1, and NaN.

In [57]:
positions_12m = signals_12m.shift(1)

print(positions_12m.head(15))

valid_positions = positions_12m.isin([-1, 0, 1]) | positions_12m.isna()
print(valid_positions.all().all())

Ticker      DBA  DBB  DBC  EEM  EWJ  FXA  FXB  FXE  FXY  HYG  IEF  LQD  SPY  \
Date                                                                          
2008-01-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-02-29  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-03-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-04-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-05-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-06-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-07-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-08-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-09-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-10-31  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN   
2008-11-30  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

## 5. Position Changes & Turnover

This section measures monthly changes in trading positions.

Position changes are used to identify trading activity and provide a preliminary measure of turnover.

- Unchanged position: no trading required.
- Long to Short (or Short to Long): absolute position change of 2.
- Long or Short to Flat: absolute position change of 1.

Missing positions remain undefined rather than being automatically replaced with zero.

Transaction costs and realistic execution assumptions will be introduced in Phase 3E.

In [58]:
positions_change = (positions_12m.diff()).abs()
total_asset_positions_change = positions_change.sum()
global_positions_change = total_asset_positions_change.sum()

print(total_asset_positions_change)
print(global_positions_change)

monthly_position_change = (positions_change.sum(axis=1, min_count=1))
print(monthly_position_change.loc["2009-01":"2009-05"])
print(monthly_position_change.loc["2010"].head())

Ticker
DBA    22.0
DBB    26.0
DBC    38.0
EEM    38.0
EWJ    38.0
FXA    44.0
FXB    36.0
FXE    40.0
FXY    42.0
HYG    30.0
IEF    22.0
LQD    20.0
SPY    22.0
TLT    30.0
USO    46.0
VGK    42.0
dtype: float64
536.0
Date
2009-01-31    NaN
2009-02-28    NaN
2009-03-31    0.0
2009-04-30    0.0
2009-05-31    0.0
Freq: ME, dtype: float64
Date
2010-01-31    6.0
2010-02-28    0.0
2010-03-31    4.0
2010-04-30    4.0
2010-05-31    2.0
Freq: ME, dtype: float64


## 6. Position Exposure Analysis

This section examines the distribution of Long, Short, and Flat positions across the ETF universe.

The objective is to understand the strategy's directional exposure over time before introducing portfolio weights.

Undefined positions are excluded from exposure statistics.

In [85]:
monthly_long_count = (positions_12m == 1).sum(axis=1)
monthly_flat_count = (positions_12m == 0).sum(axis=1)
monthly_short_count = (positions_12m == -1).sum(axis=1)

valid_months = (positions_12m.notna().sum(axis=1)) >= 1 

monthly_long_count = monthly_long_count.where(valid_months)
monthly_short_count = monthly_short_count.where(valid_months)

print(monthly_long_count.loc["2009-01":"2009-03"])
print(monthly_short_count.loc["2009-01":"2009-03"])

monthly_flat_count = monthly_flat_count.where(valid_months)

print(monthly_flat_count.loc["2009-01":"2009-03"])

monthly_total_count = (monthly_long_count + monthly_short_count + monthly_flat_count)

complete_months = positions_12m.notna().all(axis=1)

print((monthly_total_count.loc[complete_months] == 16).all())

Date
2009-01-31    NaN
2009-02-28    3.0
2009-03-31    3.0
Freq: ME, dtype: float64
Date
2009-01-31     NaN
2009-02-28    13.0
2009-03-31    13.0
Freq: ME, dtype: float64
Date
2009-01-31    NaN
2009-02-28    0.0
2009-03-31    0.0
Freq: ME, dtype: float64
True


## 7. Conclusion

### Research Summary

Phase 3D.01 established and validated the position construction framework for a 12-month Time-Series Momentum (TSMOM) strategy across 16 ETFs.

**Pipeline:** Monthly Prices → Momentum → Signals → Lagged Positions → Position Changes → Exposure Analysis.

### Key Results

- **Data:** 168 monthly observations (2008–2021), 16 ETFs, no missing prices.
- **Signals:** Long (+1), Short (-1), Flat (0).
- **Temporal alignment:** `shift(1)` prevents look-ahead bias in the monthly return convention.
- **Missing values:** Initial undefined signals and positions remain `NaN`.
- **Position changes:** 536 cumulative units across the ETF universe, excluding initial entries.
- **Validation:** All complete months satisfy the 16-asset exposure consistency check.

### Limitations

Portfolio weights, transaction costs, slippage, financing costs, and strategy performance have not yet been evaluated.

The results validate the construction methodology, **not the profitability of the strategy**.

### Next Step

**Phase 3D.02 — Portfolio Construction:** Define portfolio weights, capital allocation, and exposure constraints.

**Status: 3D.01 — Completed and Validated.**